In [1]:
import cv2
import cv2 as cv
import numpy as np
import torch
from ultralytics import YOLO
import math
from sklearn.cluster import DBSCAN
from scipy.optimize import linear_sum_assignment

from mapalignmentprocessor import MapAlignmentProcessor
from tracker import EKFTracker


WARNING Ultralytics settings reset to default values. This may be due to a possible problem with your settings or a recent ultralytics package update. 
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\agk98\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [2]:
def initial_orientation(prev_pos, curr_pos, epsilon=1e-6):
    dx = curr_pos[ 0 ] - prev_pos[ 0 ]
    dy = curr_pos[ 1 ] - prev_pos[ 1 ]
    if abs(dx) < epsilon and abs(dy) < epsilon:
        return 0.0
    return math.atan2(dy, dx)


def initial_position(prev_pos, curr_pos, dt):
    dx = curr_pos[ 0 ] - prev_pos[ 0 ]
    dy = curr_pos[ 1 ] - prev_pos[ 1 ]
    vx = dx / dt
    vy = dy / dt
    return vx, vy


def unwrap_angle(angle, prev_angle):
    diff = angle - prev_angle
    while diff > math.pi:
        diff -= 2 * math.pi
    while diff <= -math.pi:
        diff += 2 * math.pi
    return prev_angle + diff


def detect_and_get_points(model, cap, 
                          color, offset_factor=0.1):
    ret, frame = cap.read()
    if not ret:
        return None, None
    results = model(frame, conf=0.5, iou=0.5)
    points = [ ]
    for result in results:
        for box in result.boxes:
            xyxy = box.xyxy[ 0 ].cpu().numpy()
            conf = box.conf[ 0 ]
            cls = int(box.cls[ 0 ])
            class_name = model.names[ cls ]
            if class_name in VEHICLE_CLASSES:
                x1, y1, x2, y2 = map(int, xyxy)
                offset = int((y2 - y1) * offset_factor)
                bottom_center = (((x1 + x2) // 2) - offset, y2 - offset)
                points.append(bottom_center)
                cv.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 1)
                cv.putText(frame, f"{class_name}: {conf:.2f}", (x1, y1 - 5),
                           cv.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
                cv.circle(frame, (bottom_center[0], bottom_center[1]), 6, color, -1)
    return frame, points


def bbox_cluster_per_camera(detections, epsilon=10, min_samples=1):
    if len(detections) == 0:
        return [ ]
    X = np.array(detections)
    db = DBSCAN(eps=epsilon, min_samples=min_samples).fit(X)
    labels = db.labels_
    unique_labels = set(labels)
    fused_points = [ ]
    for label in unique_labels:
        if label == -1:
            noise_points = X[ labels == -1 ]
            for pt in noise_points:
                fused_points.append(tuple(pt))
        else:
            cluster = X[ labels == label ]
            centroid = np.mean(cluster, axis=0)
            fused_points.append((int(centroid[ 0 ]), int(centroid[ 1 ])))
    return fused_points


def fuse_points_one_per_camera(detections, distance_threshold):
    clusters = [ ]
    for (pt, cam_label) in detections:
        placed = False
        for cluster in clusters:
            if cam_label in cluster:
                continue
            existing_points = np.array(list(cluster.values()))
            centroid = np.mean(existing_points, axis=0)
            if np.linalg.norm(np.array(pt) - centroid) < distance_threshold:
                cluster[ cam_label ] = pt
                placed = True
                break
        if not placed:
            new_cluster = {cam_label: pt}
            clusters.append(new_cluster)
    fused_results = [ ]
    for cluster in clusters:
        points_array = np.array(list(cluster.values()))
        fused_centroid = np.mean(points_array, axis=0)
        fused_centroid = (int(fused_centroid[ 0 ]), int(fused_centroid[ 1 ]))
        camera_labels = ",".join(sorted(cluster.keys()))
        fused_results.append((fused_centroid, camera_labels))
    return fused_results


def draw_fixed_rotated_bbox(img, center, width, height, angle_rad=math.pi / 2,
                            color=(255, 255, 255), thickness=2):
    angle_deg = math.degrees(angle_rad) - 90
    rotated_rect = (center, (width, height), angle_deg)
    box_points = cv.boxPoints(rotated_rect)
    box_points = np.intp(box_points)
    cv.polylines(img, [ box_points ], isClosed=True, color=color,
                 thickness=thickness)
    arrow_length = 50
    arrow_end = (int(center[ 0 ] + arrow_length * math.cos(angle_rad)),
                 int(center[ 1 ] + arrow_length * math.sin(angle_rad)))
    cv.arrowedLine(img, center, arrow_end, (0, 0, 255), thickness,
                   tipLength=0.3)



def data_association(trackers, detections, cost_threshold=50):
    """
    trackers: dict of tracker_id -> EKFTracker
    detections: list of dicts containing detection info, e.g. {'x': x_coord, 'y': y_coord, ...}
    Returns:
      matched: dict of tracker_id -> detection (dict)
      unmatched_tracker_ids: set of tracker ids not matched
      unmatched_detection_idxs: set of indices for detections not matched
    """
    tracker_ids = list(trackers.keys())
    num_trackers = len(tracker_ids)
    num_detections = len(detections)
    if num_trackers == 0 or num_detections == 0:
        return {}, set(tracker_ids), set(range(num_detections))
    cost_matrix = np.zeros((num_trackers, num_detections), dtype=np.float32)
    for i, t_id in enumerate(tracker_ids):
        pred_state = trackers[ t_id ].x  # Use tracker instance state directly
        pred_x, pred_y = pred_state[ 0, 0 ], pred_state[ 1, 0 ]
        for j, detection in enumerate(detections):
            cost_matrix[ i, j ] = math.hypot(pred_x - detection[ "x" ],
                                             pred_y - detection[ "y" ])
    row_ind, col_ind = linear_sum_assignment(cost_matrix)
    matched = {}
    unmatched_tracker_ids = set(tracker_ids)
    unmatched_detection_idxs = set(range(num_detections))
    for r, c in zip(row_ind, col_ind):
        if cost_matrix[ r, c ] < cost_threshold:
            t_id = tracker_ids[ r ]
            matched[ t_id ] = detections[ c ]
            unmatched_tracker_ids.discard(t_id)
            unmatched_detection_idxs.discard(c)
    return matched, unmatched_tracker_ids, unmatched_detection_idxs

In [3]:
AVG_CAR_WIDTH = 50
AVG_CAR_HEIGHT = 100
VEHICLE_CLASSES = [ "car", "truck" ]
active_trackers = {}
lost_trackers = {}

max_missed_frames = 10
lost_tracker_timeout = 20
next_tracker_id = 0

model_path = "yolov8s.pt"
model = YOLO(model_path)

satellite_image_path = ("camera_data\\top_down_view\\satellite_image.png")

satellite_base = cv.imread(satellite_image_path)
if satellite_base is None:
    raise ValueError("Error: Could not load satellite image.")

cam5_video = "dataset\\video_clips\\cam_5.mp4"
cam5_other_image = "dataset\\frames\\raw\\5\\frame_0000.jpg"
cam5_target_points = "camera_data\\cam5\\ref_points.txt"
cam5_other_points = "camera_data\\cam5\\other_points.txt"

cam6_video = "dataset\\video_clips\\cam_6.mp4"
cam6_other_image = "dataset\\frames\\raw\\6\\frame_0000.jpg"
cam6_target_points = "camera_data\\cam6\\ref_points.txt"
cam6_other_points = "camera_data\\cam6\\other_points.txt"

cam7_video = "dataset\\video_clips\\cam_7.mp4"
cam7_other_image = "dataset\\frames\\raw\\camera_3\\frame_0000.jpg"
cam7_target_points = "camera_data\\cam7\\ref_points.txt"
cam7_other_points = "camera_data\\cam7\\other_points.txt"

map_processor1 = MapAlignmentProcessor(satellite_image_path,
                                       cam5_other_image,
                                       cam5_target_points,
                                       cam5_other_points,
                                       color=(0, 0, 255), cam_label="C5")
map_processor2 = MapAlignmentProcessor(satellite_image_path,
                                       cam6_other_image,
                                       cam6_target_points,
                                       cam6_other_points,
                                       color=(0, 255, 0), cam_label="C6")
map_processor3 = MapAlignmentProcessor(satellite_image_path,
                                       cam7_other_image,
                                       cam7_target_points,
                                       cam7_other_points,
                                       color=(255, 0, 0), cam_label="C7")

In [4]:
count = 0
cap5 = cv.VideoCapture(cam5_video)
cap6 = cv.VideoCapture(cam6_video)
cap7 = cv.VideoCapture(cam7_video)

active_trackers = {}
lost_trackers = {}

next_tracker_id = 0

dt = 1 / 30
P = np.eye(5) * 1.0
Q = np.eye(5)
Q[ 0, 0 ] = 0.2
Q[ 1, 1 ] = 0.2
Q[ 2, 2 ] = 0.05
Q[ 3, 3 ] = 0.05
Q[ 4, 4 ] = 0.01
R = np.eye(3)
R[ 0, 0 ] = 0.5
R[ 1, 1 ] = 0.5
R[ 2, 2 ] = 0.1

association_threshold = 60
max_missed_frames = 10
lost_tracker_timeout = 20

tracker_prev_meas = {}
sample_print = 0
while True:
    satellite_view = satellite_base.copy()

    # Get detections from each camera.
    frame1, points1 = detect_and_get_points(model, cap5, color = (0, 0,255), offset_factor=0.1)
    frame2, points2 = detect_and_get_points(model, cap6, color = (0, 255,0),offset_factor=0.1)
    frame3, points3 = detect_and_get_points(model, cap7, color=(0, 255, 255),offset_factor=0.1)
    sample_print = 1

    all_detections = [ ]
    if points1 is not None:
        clustered_points1 = bbox_cluster_per_camera(points1, epsilon=10)
        for pt in clustered_points1:
            mapped_pt = map_processor1.transform_point(pt)
            all_detections.append((mapped_pt, map_processor1.cam_label))
    if points2 is not None:
        clustered_points2 = bbox_cluster_per_camera(points2, epsilon=10)
        for pt in clustered_points2:
            mapped_pt = map_processor2.transform_point(pt)
            all_detections.append((mapped_pt, map_processor2.cam_label))
    if points3 is not None:
        clustered_points3 = bbox_cluster_per_camera(points3, epsilon=10)
        for pt in clustered_points3:
            mapped_pt = map_processor3.transform_point(pt)
            all_detections.append((mapped_pt, map_processor3.cam_label))

    # Fuse detections from multiple cameras.
    fused_detections = fuse_points_one_per_camera(all_detections,
                                                  distance_threshold=60)

    # Build a list of detection dictionaries for the data association.
    detection_dicts = [ ]
    for fused_center, label in fused_detections:
        detection_dicts.append(
            {"x": fused_center[ 0 ], "y": fused_center[ 1 ],
             "label": label})
        cv.circle(satellite_view, (fused_center[ 0 ], fused_center[ 1 ]), 6, (255, 255, 255), -1)


    for tracker in active_trackers.values():
        tracker.predict()

    # Associate active trackers with current detections.
    matched, unmatched_tracker_ids, unmatched_det_idxs = data_association(
        active_trackers, detection_dicts,
        cost_threshold=association_threshold)


    for tracker_id, detection in matched.items():
        fused_center = (detection[ "x" ], detection[ "y" ])
        # Obtain the tracker's state and compute the orientation.
        tracker_state = active_trackers[ tracker_id ].x
        if tracker_id in tracker_prev_meas:
            prev_meas = tracker_prev_meas[ tracker_id ]
            vx, vy = initial_position(prev_meas, fused_center, dt)
            ALPHA = 0.2
            active_trackers[ tracker_id ].rolling_vx = ALPHA * vx + (
                        1 - ALPHA) * active_trackers[
                                                           tracker_id ].rolling_vx
            active_trackers[ tracker_id ].rolling_vy = ALPHA * vy + (
                        1 - ALPHA) * active_trackers[
                                                           tracker_id ].rolling_vy
            raw_theta = math.atan2(active_trackers[ tracker_id ].rolling_vy,
                                   active_trackers[ tracker_id ].rolling_vx)
            speed = math.hypot(active_trackers[ tracker_id ].rolling_vx,
                               active_trackers[ tracker_id ].rolling_vy)
            if speed < 30.0:
                raw_theta = float(tracker_state[ 4, 0 ])
            theta_meas = unwrap_angle(raw_theta,
                                      float(tracker_state[ 4, 0 ]))
        else:
            theta_meas = float(tracker_state[ 4, 0 ])
        measurement = np.array(
            [ fused_center[ 0 ], fused_center[ 1 ], theta_meas ]).reshape(3,
                                                                          1)
        active_trackers[ tracker_id ].update(measurement)
        tracker_prev_meas[ tracker_id ] = fused_center

    # Increase missed_frames and move trackers to the lost pool if they exceed the threshold.
    for tracker_id in list(unmatched_tracker_ids):
        tracker = active_trackers[ tracker_id ]
        tracker.missed_frames += 1
        if tracker.missed_frames > max_missed_frames:
            tracker.lost_age = 0  # Initialize lost_age on first move.
            lost_trackers[ tracker_id ] = tracker
            del active_trackers[ tracker_id ]

    # Increment the lost_age for each lost tracker and remove those expired.
    for tracker_id in list(lost_trackers.keys()):
        lost_trackers[ tracker_id ].lost_age += 1
        if lost_trackers[ tracker_id ].lost_age > lost_tracker_timeout:
            del lost_trackers[ tracker_id ]

    # For each unmatched detection, first try to match with a lost tracker (to recover its original ID);
    # if no suitable lost tracker is found, create a new tracker.
    for idx in list(unmatched_det_idxs):
        detection = detection_dicts[ idx ]
        detection_pt = np.array([ detection[ "x" ], detection[ "y" ] ])
        found_match = False
        for lost_id, lost_tracker in list(lost_trackers.items()):
            last_state = lost_tracker.x
            lost_pt = np.array([ last_state[ 0, 0 ], last_state[ 1, 0 ] ])
            if np.linalg.norm(
                    detection_pt - lost_pt) < association_threshold:
                # Revive lost tracker: update its state and move it back to active_trackers.
                theta_meas = float(lost_tracker.x[
                                       4, 0 ])  # (You may recompute orientation if desired.)
                measurement = np.array([ detection[ "x" ], detection[ "y" ],
                                         theta_meas ]).reshape(3, 1)
                lost_tracker.update(measurement)
                active_trackers[ lost_id ] = lost_tracker
                tracker_prev_meas[ lost_id ] = (
                detection[ "x" ], detection[ "y" ])
                del lost_trackers[ lost_id ]
                found_match = True
                break
        if not found_match:
            # Create a new tracker.
            fused_center = (detection[ "x" ], detection[ "y" ])
            initial_state = np.array(
                [ fused_center[ 0 ], fused_center[ 1 ], 0.0, 0.0,
                  0.0 ]).reshape(5, 1)
            new_tracker = EKFTracker(initial_state, P.copy(), Q.copy(),
                                     R.copy(), dt)
            active_trackers[ next_tracker_id ] = new_tracker
            tracker_prev_meas[ next_tracker_id ] = fused_center
            next_tracker_id += 1

    for tracker_id, tracker in active_trackers.items():
        state = tracker.x
        pos = (int(state[ 0, 0 ]), int(state[ 1, 0 ]))
        theta = state[ 4, 0 ]
        draw_fixed_rotated_bbox(satellite_view, pos, AVG_CAR_WIDTH,
                                AVG_CAR_HEIGHT, theta)
        cv.circle(satellite_view, pos, 8, (255, 255, 255), -1)
        cv.putText(satellite_view, str(tracker_id),
                   (pos[ 0 ] + 5, pos[ 1 ] + 5),
                   cv.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
    
    count +=1
    if frame1 is not None:
        cv.imshow("Camera 5", frame1)
    if frame2 is not None:
        cv.imshow("Camera 6", frame2)
    if frame3 is not None:
        cv.imshow("Camera 7", frame3)
    cv.imshow("Satellite View", satellite_view)

    if cv.waitKey(1) & 0xFF == ord("q"):
        break

cap5.release()
cap6.release()
cap7.release()
cv.destroyAllWindows()


0: 384x640 16 cars, 1 bus, 67.2ms
Speed: 7.8ms preprocess, 67.2ms inference, 216.7ms postprocess per image at shape (1, 3, 384, 640)

0: 512x640 1 person, 17 cars, 1 truck, 86.6ms
Speed: 4.0ms preprocess, 86.6ms inference, 1.9ms postprocess per image at shape (1, 3, 512, 640)

0: 512x640 2 persons, 15 cars, 1 truck, 13.2ms
Speed: 2.5ms preprocess, 13.2ms inference, 2.2ms postprocess per image at shape (1, 3, 512, 640)

0: 384x640 16 cars, 16.7ms
Speed: 4.5ms preprocess, 16.7ms inference, 3.8ms postprocess per image at shape (1, 3, 384, 640)

0: 512x640 1 person, 18 cars, 1 truck, 17.5ms
Speed: 3.0ms preprocess, 17.5ms inference, 3.9ms postprocess per image at shape (1, 3, 512, 640)

0: 512x640 2 persons, 13 cars, 1 bus, 1 truck, 19.3ms
Speed: 3.0ms preprocess, 19.3ms inference, 7.4ms postprocess per image at shape (1, 3, 512, 640)

0: 384x640 17 cars, 12.5ms
Speed: 1.9ms preprocess, 12.5ms inference, 2.8ms postprocess per image at shape (1, 3, 384, 640)

0: 512x640 18 cars, 1 truck, 1